# PPO (Proximal Policy Optimization) Clipped Loss

**难度:** Medium

实现 PPO 裁剪代理损失。

PPO 通过裁剪重要性采样比率来约束策略更新，防止强化学习中的破坏性大幅更新。

**签名:** `ppo_loss(new_logps, old_logps, advantages, clip_ratio=0.2) -> Tensor`

**参数:**
- `new_logps` — 当前策略对数概率 (B,)
- `old_logps` — 旧策略对数概率 (B,)，视为常量
- `advantages` — 优势估计 (B,)，视为常量
- `clip_ratio` — 裁剪范围 epsilon

**返回:** 标量损失

**约束:**
- 比率：`r = exp(new_logps - old_logps.detach())`
- 损失：`-min(r * A, clamp(r, 1-eps, 1+eps) * A).mean()`
- 梯度仅通过 new_logps 流动

**提示:**
1. r = exp(new_logps - old_logps.detach())
2. unclipped = r * advantages.detach()
3. clipped = clamp(r, 1-ε, 1+ε) * advantages.detach()
4. loss = -min(unclipped, clipped).mean()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [ ]:
# ✅ SOLUTION

def ppo_loss(new_logps: Tensor, old_logps: Tensor, advantages: Tensor,
             clip_ratio: float = 0.2) -> Tensor:
    # ---- Step 1: 准备常量 ----
    # old_logps 和 advantages 是"旧数据"，不应该有梯度
    old_logps_detached = old_logps.detach()
    adv_detached = advantages.detach()

    # ---- Step 2: 计算重要性采样比率 ----
    # r = π_new / π_old，在 log 空间做减法更稳定
    # r > 1: 新策略比旧策略更倾向这个 action
    # r < 1: 新策略比旧策略更不倾向这个 action
    ratios = torch.exp(new_logps - old_logps_detached)

    # ---- Step 3: 计算裁剪目标 ----
    # unclipped = r * A（标准策略梯度目标）
    unclipped = ratios * adv_detached

    # clipped = clamp(r, 1-ε, 1+ε) * A（限制比率在 [1-ε, 1+ε] 范围内）
    # 当 r 超出范围时，用边界值替代
    clipped = torch.clamp(ratios, 1.0 - clip_ratio, 1.0 + clip_ratio) * adv_detached

    # ---- Step 4: 取两者的最小值 ----
    # 为什么取 min？
    # 当 A > 0（好的 action）：min 限制了 r 的上界 → 不能因为这个 action 太好就过度增加其概率
    # 当 A < 0（差的 action）：min 限制了 r 的下界 → 不能因为这个 action 太差就过度降低其概率
    # 效果：无论 A 正负，策略更新幅度都被约束在 clip_ratio 范围内
    return -torch.min(unclipped, clipped).mean()

In [ ]:
# Demo
new_logps = torch.tensor([0.0, -0.2, -0.4, -0.6])
old_logps = torch.tensor([0.0, -0.1, -0.5, -0.5])
advantages = torch.tensor([1.0, -1.0, 0.5, -0.5])
print('Loss:', ppo_loss(new_logps, old_logps, advantages, clip_ratio=0.2))


In [ ]:
from torch_judge import check
check('ppo_loss')


## 📝 核心思路总结

1. **PPO 的核心：裁剪约束策略更新幅度**：防止新策略偏离旧策略太远导致训练崩溃
2. **重要性采样比率 r = π_new/π_old**：用旧数据估计新策略的梯度（off-policy）
3. **min 取保守方向**：当 A>0 时 clip 上界，A<0 时 clip 下界，确保更新幅度有界